# Diffract Visualization Showcase

This notebook is a **practical end-to-end example** of `diffract`:

- Build a realistic toy Transformer with `layer_id` / `head_id` metadata
- Run computations via the public `Session` API
- Render multiple plot types via `diffract.viz` (including Hydra-configured plots)
- Apply theming + color control
- **Save every generated figure into `examples/images/` as SVG** (fallback to PNG)

All YAML configs used by this notebook are located in `examples/configs/`.

In [ ]:
!pip install hydra-core

In [ ]:
!plotly_get_chrome -y

In [ ]:
import rootutils
from pathlib import Path
import re
import sys

import plotly.io as pio

root = rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
sys.path.insert(0, str(root / "src"))

# Prefer SVG in notebook outputs.
pio.renderers.default = "svg"

# Example assets live in examples/.
EXAMPLES_DIR = root / "examples"
CONFIGS_DIR = EXAMPLES_DIR / "configs"
IMAGES_DIR = EXAMPLES_DIR / "images"
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def _safe_stem(name: str) -> str:
    out = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", "."):
            out.append(ch)
        else:
            out.append("_")
    return "".join(out).strip("_")


def save_fig(fig, name: str) -> Path:
    """Save figure into examples/images as PNG.

    Note: this repo's assistant tooling can reliably inspect PNG outputs.
    """
    stem = _safe_stem(name)
    png_path = IMAGES_DIR / f"{stem}.png"
    fig.write_image(png_path, scale=2)
    return png_path


def show_and_save(fig, name: str) -> None:
    fig.show()
    out = save_fig(fig, name)
    try:
        rel = out.relative_to(root)
    except Exception:
        rel = out
    print(f"saved: {rel}")


import torch
import torch.nn as nn
from diffract import ParameterOverrides, Session

## 1. Define Neural Network Models

In [ ]:
class ToyAttentionHead(nn.Module):
    def __init__(self, d_model: int, d_head: int):
        super().__init__()
        # One projection per head (keeps extraction simple and gives us head_id).
        self.proj = nn.Linear(d_model, d_head, bias=False)

    def forward(self, x):
        return self.proj(x)


class ToyTransformerLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_head: int, d_ff: int):
        super().__init__()
        self.heads = nn.ModuleList([ToyAttentionHead(d_model, d_head) for _ in range(n_heads)])
        # One FFN per layer (not head-specific).
        self.ffn = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        # Not used by diffract; kept for completeness.
        heads_out = [h(x) for h in self.heads]
        return self.ffn(x) + sum(heads_out)


class ToyTransformer(nn.Module):
    def __init__(self, d_model: int, n_layers: int, n_heads: int, d_head: int, d_ff: int):
        super().__init__()
        self.layers = nn.ModuleList(
            [ToyTransformerLayer(d_model, n_heads, d_head, d_ff) for _ in range(n_layers)]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


def build_toy_overrides(model: nn.Module) -> dict[str, ParameterOverrides]:
    """Attach layer/head metadata derived from module names."""
    overrides: dict[str, ParameterOverrides] = {}

    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue

        m = re.match(r"^layers\.(\d+)\.heads\.(\d+)\.proj$", name)
        if m:
            overrides[name] = ParameterOverrides(
                other_meta={
                    "layer_id": int(m.group(1)),
                    "head_id": int(m.group(2)),
                    "kind": "attn_proj",
                }
            )
            continue

        m = re.match(r"^layers\.(\d+)\.ffn$", name)
        if m:
            overrides[name] = ParameterOverrides(
                other_meta={
                    "layer_id": int(m.group(1)),
                    "head_id": None,
                    "kind": "ffn",
                }
            )

    return overrides


model1 = ToyTransformer(d_model=256, n_layers=2, n_heads=4, d_head=64, d_ff=512)
model2 = ToyTransformer(d_model=512, n_layers=4, n_heads=8, d_head=64, d_ff=1024)

print(f"ToyTransformer small: {sum(p.numel() for p in model1.parameters())} params")
print(f"ToyTransformer big: {sum(p.numel() for p in model2.parameters())} params")

## 2. Initialize Session and Add Models

In [ ]:
session = Session()

with session:
    print(f"Kernels: {len(session.compute.list_available_kernels())}")
    session.models.add(
        model1,
        model_id="toy_small",
        parameter_overrides=build_toy_overrides(model1),
    )
    session.models.add(
        model2,
        model_id="toy_big",
        parameter_overrides=build_toy_overrides(model2),
    )
    print(f"Parameters: {len(session.models.parameters.list())}")

## 3. Compute Fields

In [ ]:
with session:
    for field in ["frob_norm", "effective_rank", "stable_rank"]:
        session.compute.apply(field)
        print(f"✓ {field}")

## 3b. Plot: stable_rank boxplot

This uses `diffract.viz` (public `Session` API only) to render a boxplot of `stable_rank` grouped by `model_id`.

In [ ]:
from diffract.viz.data import FieldRef
from diffract.viz.plots.boxplot import BoxPlot

with session:
    fig = session.viz.draw(
        plot=BoxPlot(
            y=FieldRef("stable_rank"),
            title="stable_rank by model_id",
            x=FieldRef("model_id"),
        )
    )
    show_and_save(fig, "3b_stable_rank_boxplot")

# Same plot rendered from a YAML config (Hydra `_target_`) with optional overrides.
# This is useful for keeping complex plot definitions declarative.
with session:
    fig2 = session.viz.draw(
        config_path=CONFIGS_DIR / "boxplot_stable_rank.yaml",
        overrides=[],
    )
    show_and_save(fig2, "3b_stable_rank_boxplot_from_yaml")

## 3c. Plot: scatter (stable_rank vs frob_norm)

A generic scatter plot based on two scalar fields, grouped by `model_id`.

In [ ]:
from diffract.viz.data import FieldRef
from diffract.viz.plots.scatter import ScatterPlot

with session:
    fig = session.viz.draw(
        plot=ScatterPlot(
            x=FieldRef("frob_norm"),
            y=FieldRef("stable_rank"),
            title="stable_rank vs frob_norm",
            group_by=FieldRef("model_id"),
        )
    )
    show_and_save(fig, "3c_scatter_stable_rank_vs_frob_norm")

## 3d. Plot: heatmap pivot (stable_rank by layer_id × head_id)

A generic heatmap built by pivoting a scalar field using metadata keys.

- `y="layer_id"` (rows)
- `x="head_id"` (columns)

Only the attention-head projections carry `head_id`, so the session is filtered to those parameters.

In [ ]:
from diffract.viz.data import FieldRef
from diffract.viz.plots.heatmap import HeatmapPlot

filtered = session.filter(model_ids=["toy_big"], param_names=["re:.*heads.*"])
with filtered:
    fig = filtered.viz.draw(
        plot=HeatmapPlot(
            z=FieldRef("stable_rank"),
            y=FieldRef("layer_id"),
            x=FieldRef("head_id"),
            title="stable_rank heatmap (layer_id × head_id)",
            show_text=False,
            heatmap_colorscale="Viridis",
        )
    )
    show_and_save(fig, "3d_heatmap_stable_rank_model_by_param")

## 3e. Plot: line-by-metadata (frob_norm by in_model_idx)

A generic “sparkline-like” plot: scalar metric vs a metadata key (x-axis).

This expects `in_model_idx` to exist in exported metadata (it comes from extractor metadata for this notebook models).

In [ ]:
from diffract.viz.data import FieldRef
from diffract.viz.plots.sparkline import SparklinePlot

with session:
    fig = session.viz.draw(
        plot=SparklinePlot(
            y=FieldRef("frob_norm"),
            x=FieldRef("in_model_idx"),
            title="frob_norm by in_model_idx (grouped by model_id)",
            group_by=FieldRef("model_id"),
            mode="lines+markers",
        )
    )
    show_and_save(fig, "3e_line_frob_norm_by_in_model_idx")

## 3f. Plot customization: UpdateFigure

`UpdateFigure` lets you keep plots minimal while still having full Plotly customization (layout / traces / axes) in one place.

In [ ]:
from diffract.viz.data import FieldRef
from diffract.viz.plots.base import UpdateFigure
from diffract.viz.plots.scatter import ScatterPlot

with session:
    fig = session.viz.draw(
        plot=UpdateFigure(
            plot=ScatterPlot(
                x=FieldRef("frob_norm"),
                y=FieldRef("stable_rank"),
                title=None,  # set via layout below
                group_by=FieldRef("model_id"),
            ),
            layout=dict(
                title="stable_rank vs frob_norm (customized)",
                width=800,
                height=400,
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            ),
            traces=dict(marker=dict(size=9, opacity=0.6), mode="markers"),
            xaxes=dict(title="frob_norm", showgrid=True, zeroline=False),
            yaxes=dict(title="stable_rank", showgrid=True, zeroline=False),
        )
    )
    show_and_save(fig, "3f_updatefigure_scatter_customized")

## 3g. Plot: violins (weights_svals) + jitter colored by field

A universal violin plot with a configurable jitter overlay.

Here we color jitter points by `jitter_color_field` (same field for demo, but it can be any compatible field).

In [ ]:
with session:
    # weights_svals is required by the violin example
    session.compute.apply("weights_svals")

with session:
    fig = session.viz.draw(
        config_path=CONFIGS_DIR / "violin_weights_svals_jitter.yaml",
        overrides=[],
    )
    show_and_save(fig, "3g_violin_weights_svals_jitter")

## 3h. Plot: boxplot + jitter

A box plot of a scalar field with a configurable jitter overlay, colored by `jitter_color_field`.

In [ ]:
with session:
    # stable_rank and frob_norm are required by this example
    session.compute.apply("stable_rank", "frob_norm")

with session:
    fig = session.viz.draw(
        config_path=CONFIGS_DIR / "boxplot_stable_rank_jitter.yaml",
        overrides=[],
    )
    show_and_save(fig, "3h_boxplot_stable_rank_jitter")

## 3j. Subplots

In [ ]:
with session:
    fig = session.viz.draw(
        config_path=CONFIGS_DIR / "grid_example.yaml",
        overrides=[],
    )
    show_and_save(fig, "3j_grid_example")

## 3k. Theming and Color Customization

Plots support a unified theming system for publication-ready figures.
- Pass a `Theme` to `session.viz.draw(..., theme=DARK_THEME)` (built-ins: `DEFAULT_THEME`, `DARK_THEME`, `MINIMAL_THEME`, all re-exported from `diffract.viz`)
- Color by any metadata key with `marker_color=FieldRef("layer_id")`
- Load a theme from YAML with `theme_path=...`

In [ ]:
from diffract.viz import DARK_THEME, MINIMAL_THEME
from diffract.viz.data import FieldRef
from diffract.viz.plots.boxplot import BoxPlot

# Plot with theme (passed to draw) + coloring by metadata (layer_id).
# marker_color drives the color axis; the jitter overlay renders the
# individual points so the per-layer coloring is visible.
with session:
    fig = session.viz.draw(
        plot=BoxPlot(
            y=FieldRef("stable_rank"),
            title="Stable Rank by Model (Theme + Color by layer_id)",
            x=FieldRef("model_id"),
            boxpoints=False,
            marker_color=FieldRef("layer_id"),
            jitter_enabled=True,
        ),
        theme=DARK_THEME,
    )
    show_and_save(fig, "3k_theme_color_by_layer")

# Same plot with minimal theme
with session:
    fig2 = session.viz.draw(
        plot=BoxPlot(
            y=FieldRef("stable_rank"),
            title="Stable Rank (Minimal Theme)",
            x=FieldRef("model_id"),
        ),
        theme=MINIMAL_THEME,
    )
    show_and_save(fig2, "3k_theme_minimal")

# Load theme from YAML file
with session:
    fig3 = session.viz.draw(
        config_path=CONFIGS_DIR / "boxplot_stable_rank.yaml",
        theme_path=CONFIGS_DIR / "theme_example.yaml",
    )
    show_and_save(fig3, "3k_theme_from_yaml")

## 5. List Models, Parameters, and Kernels

In [ ]:
with session:
    print(f"Models: {session.models.list()}")
    print(f"Parameters: {len(session.models.parameters.list())}")
    print(f"Kernels: {session.compute.list_available_kernels()[:10]}...")

## 6. Erase All Data

In [ ]:
with session:
    print(f"Before: Models={session.models.list()}, Params={len(session.models.parameters.list())}")
    session.results.erase(erase_all=True)
    print(f"After: Models={session.models.list()}, Params={len(session.models.parameters.list())}")
    session.models.erase(erase_all=True)
    print(f"After: Models={session.models.list()}, Params={len(session.models.parameters.list())}")

In [ ]:
del session